In [1]:
!git clone https://github.com/facebookresearch/sapiens.git

fatal: destination path 'sapiens' already exists and is not an empty directory.


In [ ]:
# Quiet setup: the full output goes to setup.log, only one line per step is printed
!conda env list | grep -q "^sapiens " || conda create -n sapiens python=3.10 -y -q > setup.log 2>&1 && echo "env: ok" || echo "env: FAILED (see setup.log)"
!conda run -n sapiens python -m pip install -q ipykernel >> setup.log 2>&1 && echo "ipykernel: ok" || echo "ipykernel: FAILED (see setup.log)"
!conda run -n sapiens python -m ipykernel install --user --name sapiens --display-name "Python (Sapiens)" >> setup.log 2>&1 && echo "kernel: ok" || echo "kernel: FAILED (see setup.log)"
!conda run -n sapiens python -m pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu128 >> setup.log 2>&1 && echo "torch: ok" || echo "torch: FAILED (see setup.log)"
!conda run -n sapiens python -m pip install -q numpy opencv-python pandas matplotlib huggingface_hub >> setup.log 2>&1 && echo "packages: ok" || echo "packages: FAILED (see setup.log)"

In [4]:
from huggingface_hub import login
from huggingface_hub import model_info

login()

info = model_info("noahcao/sapiens-pose-coco")

print("Authenticated successfully!")
print("Model:", info.id)

Authenticated successfully!
Model: noahcao/sapiens-pose-coco


In [13]:
from pathlib import Path

SAPIENS_ROOT = Path("sapiens").resolve()
LITE_ROOT = SAPIENS_ROOT / "lite"
SCRIPT_DIR = LITE_ROOT / "scripts" / "demo" / "torchscript"

script = SCRIPT_DIR / "pose_keypoints133.sh"

[FOUND] /home/ausjak2/saugios-rankos/sapiens/lite/demo/vis_pose.py
[MISSING] /home/ausjak2/saugios-rankos/sapiens/lite/pose/demo/mmdetection_cfg/rtmdet_m_640-8xb32_coco-person_no_nms.py
[FOUND] /home/ausjak2/saugios-rankos/sapiens/lite/scripts/demo/torchscript/pose_keypoints133.sh
[MISSING] /home/ausjak2/sapiens_lite_host/torchscript/pose/checkpoints/sapiens_1b/sapiens_1b_coco_wholebody_best_coco_wholebody_AP_727_torchscript.pt2
[MISSING] /home/ausjak2/sapiens_lite_host/torchscript/detector/checkpoints/rtmpose/rtmdet_m_8xb32-100e_coco-obj365-person-235e8209.pth


In [14]:
from huggingface_hub import hf_hub_download
from pathlib import Path

CHECKPOINT_ROOT = (
    Path.home() / "sapiens_lite_host" / "torchscript"
)

MODEL_DIR = (
    CHECKPOINT_ROOT / "pose" / "checkpoints" / "sapiens_1b"
)

MODEL_DIR.mkdir(parents=True, exist_ok=True)

FILENAME = (
    "sapiens_1b_coco_wholebody_best_coco_wholebody_AP_727_torchscript.pt2"
)

MODEL_PATH = hf_hub_download(
    repo_id="noahcao/sapiens-pose-coco",
    filename=(
        "sapiens_lite_host/torchscript/pose/checkpoints/"
        f"sapiens_1b/{FILENAME}"
    ),
    local_dir=str(Path.home() / "sapiens_download")
)

Downloaded: /home/ausjak2/sapiens_download/sapiens_lite_host/torchscript/pose/checkpoints/sapiens_1b/sapiens_1b_coco_wholebody_best_coco_wholebody_AP_727_torchscript.pt2


In [ ]:
# use the downloaded checkpoint directly (works on any machine)
DEST = Path(MODEL_PATH)
print("Checkpoint:", DEST)
print(f"Size: {DEST.stat().st_size / 1e9:.2f} GB")

In [7]:
DATASET_ROOT = Path("dataset")

VIDEO_INFO_DIR = DATASET_ROOT / "video_info" / "video_info"

RESULTS_DIR = Path("sapiens_results")   # raw Sapiens output (not in git)


In [8]:
from degradations import load_setup, select_videos

# The same file as the MediaPipe notebook reads: which videos, the preprocessing every video gets first, which tests to
# run, the random seeds. Both models therefore process exactly the same videos (edit eval_setup.txt, not this cell).
SETUP_FILE = "eval_setup.txt"
SETUP      = load_setup(SETUP_FILE)
videos     = select_videos(SETUP["videos"])    # [{"sign", "video" = "<sign>__<name>", "path"}]
print(f"{len({v['sign'] for v in videos})} signs, {len(videos)} videos")


In [9]:
info_files = [
    f for f in VIDEO_INFO_DIR.rglob("*")
    if f.is_file()
]

print(f"Found {len(info_files)} information files")

for f in info_files[:30]:
    print(f.name, f.suffix)

Found 0 information files


## Sapiens inference on the shared videos

The videos, the preprocessing, the tests and the random seeds come from `eval_setup.txt` and `tests.txt`, the same files the
MediaPipe notebook reads (`degradations.py` makes the frames), so both models get **exactly the same frames**. Every video is
first preprocessed (`preprocess`, 720p / 30 fps by default, as for MediaPipe), then each scenario's filters are applied.
The original (preprocessed) videos run first, then every test in `RUN_TESTS`.

Same person boxes as SMPLest-X: SMPLest-X's own detection step is repeated (its `main/inference.py` with default config:
YOLOv8x, class person, confidence 0.5, first box, image given as RGB float), and the box goes into Sapiens' own pose code
from `sapiens/lite/demo/vis_pose.py` (`preprocess_pose` crops it with Sapiens' standard padding, `batch_inference_topdown`,
`img_save_and_vis` decodes and saves). Every frame is used, like SMPLest-X.
Frames without a detected person get no keypoints (empty `instance_info`), like SMPLest-X.

Output per frame in `sapiens_results/prep_<hash>/<scenario>/<sign>/`: `<video>_<frame>.json` (133 keypoints + scores) and a
drawn `.jpg`; per video `<video>_boxes.json` with the detector boxes. `<hash>` identifies the preprocessing, so results made
with another `preprocess` never mix. Finished videos are skipped, so the cell can be rerun after an interruption.
To process the whole dataset at its original resolution, set `"signs": null, "max_signs": null, "videos_per_sign": null`
and `"preprocess": []` in `eval_setup.txt`.


In [ ]:
%pip install -q json_tricks tqdm ultralytics imageio-ffmpeg
import sys, json, shutil, subprocess, cv2, numpy as np, torch, imageio_ffmpeg
from ultralytics import YOLO

sys.path.insert(0, str(LITE_ROOT / "demo"))      # Sapiens' own pose code
from vis_pose import load_model, preprocess_pose, batch_inference_topdown, img_save_and_vis
from classes_and_palettes import COCO_WHOLEBODY_KPTS_COLORS, COCO_WHOLEBODY_SKELETON_INFO
from degradations import load_tests, make_scenarios, scenario_folder, scenario_frames, short_hash, ORIGINAL

TF32       = True          # TF32 matrix multiplication: ~1.4x faster, mean keypoint change 0.04 px vs fp32 (measured)
BATCH_SIZE = 8             # frames per forward pass, lower it if CUDA runs out of memory
DETECTOR   = "yolov8x.pt"  # SMPLest-X default person detector (downloaded automatically)
DET_CONF   = 0.5           # SMPLest-X default detection confidence
FFMPEG     = shutil.which("ffmpeg") or imageio_ffmpeg.get_ffmpeg_exe()   # only for the GIFs below

# Shared with the MediaPipe notebook (SETUP and the video list were loaded above)
PREPROCESS   = SETUP["preprocess"]        # every video gets this first, then the scenario's filters
RUN_TESTS    = SETUP["run_tests"]         # names from TESTS; [] = only the original videos
RANDOM_SEEDS = SETUP["random_seeds"]      # noise, frame_drop and occlusion run once per seed
TESTS_FILE   = "tests.txt"
TESTS        = load_tests(TESTS_FILE)
scenarios    = make_scenarios(TESTS, RUN_TESTS, RANDOM_SEEDS)   # scenarios[0] is the original video
print(f"{len(scenarios)} scenarios (original + {len(scenarios) - 1} degraded)")

SAPIENS_OUT = RESULTS_DIR / f"prep_{short_hash(PREPROCESS)}"    # one folder per scenario inside
OUTPUT_DIR  = SAPIENS_OUT / scenario_folder(ORIGINAL)           # raw output of the original (preprocessed) videos

# Sapiens settings, as in vis_pose.py / pose_keypoints133.sh
INPUT_SHAPE, HEATMAP_SCALE, KPT_THR, RADIUS = (1024, 768), 4, 0.3, 3
MEAN, STD = [123.5, 116.5, 103.5], [58.5, 57.0, 57.5]

torch.backends.cuda.matmul.allow_tf32 = TF32
detector = YOLO(DETECTOR)
model = load_model(str(DEST), use_torchscript=True).cuda()


In [ ]:
def detect(frame):
    """SMPLest-X's detection step (main/inference.py): RGB float image, person class, first box (xyxy)."""
    boxes = detector.predict(frame[:, :, ::-1].astype(np.float32), classes=0, conf=DET_CONF, verbose=False)[0].boxes.xyxy
    return boxes[0].cpu().numpy() if len(boxes) else None


def run_video(video, out, scenario):
    stem = Path(video["path"]).stem
    # the same frames the MediaPipe notebook uses: preprocessing, then the scenario's filters
    frames, fps = scenario_frames(video["path"], video["video"], PREPROCESS, scenario)
    boxes = [detect(f) for f in frames]
    found = [t for t, b in enumerate(boxes) if b is not None]
    for start in range(0, len(found), BATCH_SIZE):
        ts = found[start:start + BATCH_SIZE]
        prep = [preprocess_pose(frames[t], [boxes[t]], INPUT_SHAPE, MEAN, STD) for t in ts]
        heatmaps = batch_inference_topdown(model, torch.stack([p[0][0] for p in prep]), dtype=torch.float32)
        for t, (_, centres, scales), heatmap in zip(ts, prep, heatmaps):
            img_save_and_vis(frames[t].copy(), {"heatmaps": [heatmap], "centres": centres, "scales": scales},
                             str(out / f"{stem}_{t:05d}.jpg"), INPUT_SHAPE[::-1], HEATMAP_SCALE,
                             COCO_WHOLEBODY_KPTS_COLORS, KPT_THR, RADIUS, COCO_WHOLEBODY_SKELETON_INFO, RADIUS)
    for t, b in enumerate(boxes):
        if b is None:                     # no person: no keypoints, original frame as picture
            json.dump({"instance_info": []}, open(out / f"{stem}_{t:05d}.json", "w"))
            cv2.imwrite(str(out / f"{stem}_{t:05d}.jpg"), frames[t])
    json.dump([None if b is None else b.tolist() for b in boxes], open(out / f"{stem}_boxes.json", "w"))
    return len(frames), len(frames) - len(found)


for s in scenarios:                       # "original" comes first
    for i, video in enumerate(videos, 1):
        out = SAPIENS_OUT / scenario_folder(s) / video["sign"]
        out.mkdir(parents=True, exist_ok=True)
        if (out / f"{Path(video['path']).stem}_boxes.json").exists():
            continue                      # finished earlier
        n, missing = run_video(video, out, s)
        print(f"[{s['scenario']}] {i}/{len(videos)} {video['sign']}/{Path(video['path']).name}: {n} frames, {missing} without a person")


### Check the detector against SMPLest-X

Same detector should give the same frames without a person, and the SMPLest-X keypoints that lie within the image should lie inside the boxes (2 px tolerance).
This compares boxes with SMPLest-X's keypoints, which are in the original frames' coordinates, so it only runs when `preprocess` in `eval_setup.txt` is `[]`.


In [ ]:
import pandas as pd
if PREPROCESS:
    print('Skipped: the boxes are in the preprocessed frames, SMPLest-X in the original frames. Set "preprocess": [] in eval_setup.txt to run this check.')
else:
    rows = []
    for video in videos:
        stem = Path(video["path"]).stem
        boxes_file = OUTPUT_DIR / video["sign"] / f"{stem}_boxes.json"
        ref_file = Path("keypoints/kp_smplestx") / video["sign"] / f"{stem}.npz"
        if not boxes_file.exists() or not ref_file.exists():
            continue
        boxes = json.load(open(boxes_file))
        ref = np.load(ref_file)
        ours, theirs = np.array([b is None for b in boxes]), ref["conf"][:, 0] == 0
        h, w = ref["image_hw"]
        inside = []
        for kp, b in zip(ref["kp2d"], boxes):     # only SMPLest-X joints within the image (hands below it can't be in any box)
            kp = kp[(kp[:, 0] >= 0) & (kp[:, 0] < w) & (kp[:, 1] >= 0) & (kp[:, 1] < h)]
            if b is not None and len(kp):
                inside.append(((kp[:, 0] >= b[0] - 2) & (kp[:, 0] <= b[2] + 2) & (kp[:, 1] >= b[1] - 2) & (kp[:, 1] <= b[3] + 2)).mean())
        rows.append({"video": Path(video["path"]).name, "frames": len(boxes), "no person (ours)": int(ours.sum()),
                     "no person (SMPLest-X)": int(theirs.sum()), "same frames": bool((ours == theirs).all()),
                     "SMPLest-X joints in image inside box": float(np.mean(inside)) if inside else float("nan")})
    check = pd.DataFrame(rows)
    print(f"{len(check)} videos | missing frames identical in {check['same frames'].mean():.1%} of videos | "
          f"SMPLest-X joints (within the image) inside the boxes: {check['SMPLest-X joints in image inside box'].mean():.1%}")
    check[~check["same frames"]]


## Look at the result: one sign, 3 different people

In [ ]:
import pandas as pd
from IPython.display import Image, display

SHOW_SIGN = videos[0]["sign"]   # sign folder name

info = pd.read_csv(VIDEO_INFO_DIR / "all_videos.txt")
done = {p.stem[:-len("_boxes")] for p in (OUTPUT_DIR / SHOW_SIGN).glob("*_boxes.json")}
people = info[info["Video file"].str[:-4].isin(done)].drop_duplicates("Participant ID").head(3)
for _, row in people.iterrows():
    stem = row["Video file"][:-4]
    gif = RESULTS_DIR / f"{stem}_sapiens.gif"
    subprocess.run([FFMPEG, "-loglevel", "error", "-y", "-framerate", "30", "-i", OUTPUT_DIR / SHOW_SIGN / f"{stem}_%05d.jpg",
                    "-vf", "fps=15,scale=400:-1", gif], check=True)
    print(row["Participant ID"], stem)
    display(Image(filename=gif))


## Keypoints for the classifier

Runs `sapiens_to_keypoints.ipynb`: the .json files become `keypoints/kp_sapiens_<scenario>/<sign>/<video>.npz`, one folder
per scenario (`kp_sapiens_original`, `kp_sapiens_resolution_96`, ...), like `kp_mediapipe_<scenario>`, all with the same 52 joints as SMPLest-X.
Then run `model_comparison.ipynb`: one classifier trained on the clean keypoints of both models, tested on every degraded scenario.


In [ ]:
%run sapiens_to_keypoints.ipynb